In [1]:
import sys
import os
import time
import joblib

import numpy as np
import pandas as pd
from tslearn.metrics import cdist_dtw

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.helper_functions import normalize_series # calcula a distância entre séries temporais usando DTW
from utils.constants import Configs

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


### Load Data From cache

In [2]:
cache_path = Configs.ICATER_WRIST_ONLY_FILTERED_DATA_DIR
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

df = pd.read_csv(cache_path)

## Calculate Constrained DTW
- Dependant DTW (DTWD).
- Euclidean Distance. 
- global warping constraint - Sakoe Chiba Tunning

Instead of tuning the DTW globally, I'll test diferent window_percentages and save them on cache. When LOSO is being performed, inside each fold there will be a selection of the best DTW based on the different saved DTW matrices. I can perform a tuning without exponential increasing the computational time of the LOSO cross-validation.

In [3]:
group_array = ['x', 'y', 'z']
sample_size = None

In [4]:
precompute_times = {} 
for movement in ["Pick", "Move", "Place"]:
    print(f"\n--- Tuning Constrained DTW for {movement} ---")
    
    normalized_data = normalize_series(df_input=df, movLabel_filter=movement, useScaler=False,
        group_array=group_array, sample_size=sample_size)
    
    seq_length = normalized_data.shape[1]
    precompute_times[movement] = {}
    
    best_score = -1
    best_radius = None
    best_time = 0.0

    print(f"  -> Computing DTW matrix for {movement} with sequence length={seq_length} and sample size={sample_size}")
    for sec in Configs.DTW_RADIUS_Seconds:
        radius = max(1, round(sec * Configs.FS))  # Convert seconds to samples, ensuring at least 1 sample
        
        t0 = time.perf_counter()
        
        dtw_matrix = cdist_dtw(normalized_data, global_constraint="sakoe_chiba", 
                                sakoe_chiba_radius=radius, n_jobs=-1)
        
        precompute_times[movement][sec] = time.perf_counter() - t0

        # Export matrix
        cache_name = Configs.wrist_only_constrained_dtw_distance_matrix(movement, sec)
        joblib.dump(dtw_matrix, cache_name)

        print(f"  -> w={sec:.2f} | radius={radius} | {precompute_times[movement][sec]:.2f}s")        


--- Tuning Constrained DTW for Pick ---


Processing time series:   0%|          | 0/380 [00:00<?, ?it/s]

Processing time series: 100%|██████████| 380/380 [00:00<00:00, 2674.41it/s]

Shape of first series: (111, 3)


Shape of formatted series: (380, 241, 3)
  -> Computing DTW matrix for Pick with sequence length=241 and sample size=None
  -> w=0.10 | radius=6 | 52.01s
  -> w=0.20 | radius=12 | 48.56s
  -> w=0.30 | radius=18 | 45.98s
  -> w=0.40 | radius=24 | 47.03s
  -> w=0.50 | radius=30 | 43.52s

--- Tuning Constrained DTW for Move ---


Processing time series: 100%|██████████| 95/95 [00:00<00:00, 1329.70it/s]

Shape of first series: (72, 3)
Shape of formatted series: (95, 196, 3)
  -> Computing DTW matrix for Move with sequence length=196 and sample size=None


  -> w=0.10 | radius=6 | 2.21s
  -> w=0.20 | radius=12 | 2.08s
  -> w=0.30 | radius=18 | 2.14s
  -> w=0.40 | radius=24 | 2.14s
  -> w=0.50 | radius=30 | 2.34s

--- Tuning Constrained DTW for Place ---


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 3183.51it/s]

Shape of first series: (290, 3)


Shape of formatted series: (380, 290, 3)
  -> Computing DTW matrix for Place with sequence length=290 and sample size=None
  -> w=0.10 | radius=6 | 54.07s
  -> w=0.20 | radius=12 | 53.65s
  -> w=0.30 | radius=18 | 54.03s
  -> w=0.40 | radius=24 | 53.82s
  -> w=0.50 | radius=30 | 54.12s
